<a href="https://colab.research.google.com/github/treborskrub/Fundamental-Cores/blob/main/routingegnv4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

"""
========================================================================
3D PATH-ROUTING CO-OPTIMIZATION ENGINE — v4
Self-Evaluating via Embedded Shortfall Engine (Production)
========================================================================
Robert's framework:
  π  — clock phase / cyclical operator
  φ  — golden ratio / growth-scaling operator
  1/3 — gate activation threshold (irreducible remainder)

Shortfall Engine (QTELLLQ-derived):
  Coherence Ledger    — self-referential budget tracking (Heisenberg overhead)
  Threshold Sentinel  — tiered SQ emission (soft/hard/critical)
  Phase Gate          — 4-state machine (Nominal/Degraded/Critical/Recovery)
  Recovery Orchestrator — minimal-cost φ-anchored intervention
  Closure Auditor     — geometric contraction test, λ sustained < 0.95
========================================================================
"""

import numpy as np
from dataclasses import dataclass, field
from typing import List, Optional
from enum import Enum

PI  = np.pi
PHI = (1 + 5**0.5) / 2
GATE_THRESHOLD = 1 / 3   # irreducible remainder: core activation rule

# ── Shortfall Engine Components ────────────────────────────────────────

class Phase(Enum):
    NOMINAL  = "NOMINAL"
    DEGRADED = "DEGRADED"
    CRITICAL = "CRITICAL"
    RECOVERY = "RECOVERY"

@dataclass
class ShortfallQuantum:
    """Atomic deficiency token. The SQ is the irreducible carrier of gap information."""
    tick: int
    severity: float
    resource: str
    phase_at_emission: Phase

@dataclass
class CoherenceLedger:
    """
    Self-referential budget tracker.
    Prices the cost of observation into the same ledger it monitors.
    Prevents monitoring-induced decoherence spiral (Heisenberg overhead).
    """
    balance: float = 1.0
    observation_cost: float = 0.0008
    history: List[float] = field(default_factory=list)

    def observe(self) -> float:
        self.balance = max(0.0, self.balance - self.observation_cost)
        self.history.append(self.balance)
        return self.balance

    def restore(self, amount: float):
        self.balance = min(1.0, self.balance + amount)

    def trend(self, window: int = 20) -> float:
        h = self.history[-window:]
        return (h[-1] - h[0]) if len(h) >= 2 else 0.0

class ThresholdSentinel:
    """
    Observation-first detection. Tiered thresholds.
    Observation frequency adapts to phase (sparse in Nominal, dense in Critical).
    """
    SOFT     = 0.15
    HARD     = 0.28
    CRITICAL_T = 0.45

    # Adaptive observation: skip ratio by phase (reduces Heisenberg overhead)
    SKIP = {Phase.NOMINAL: 3, Phase.DEGRADED: 2, Phase.CRITICAL: 1, Phase.RECOVERY: 1}

    def __init__(self, ledger: CoherenceLedger):
        self.ledger = ledger
        self.sq_window: List[ShortfallQuantum] = []
        self._tick_count = 0

    def evaluate(self, tick: int, shortfall: float, phase: Phase) -> Optional[ShortfallQuantum]:
        self._tick_count += 1
        if self._tick_count % self.SKIP.get(phase, 1) != 0:
            return None  # skip this tick (adaptive sparse schedule)

        coherence = self.ledger.observe()
        if coherence < 0.01:
            return None  # ledger critically depleted — suspend observation

        if shortfall >= self.SOFT:
            sq = ShortfallQuantum(tick, shortfall, "routing_capacity", phase)
            self.sq_window.append(sq)
            return sq
        return None

    def sq_rate(self, window: int = 40) -> float:
        return len(self.sq_window[-window:]) / window

class PhaseGate:
    """
    State machine: Nominal ↔ Degraded → Critical → Recovery → Nominal
    Exit from Recovery is controlled exclusively by the Closure Auditor.
    """
    def __init__(self):
        self.current = Phase.NOMINAL
        self.history: List[Phase] = []
        self._degraded_ticks = 0

    def evaluate(self, sq_rate: float, ledger_trend: float) -> Phase:
        if self.current == Phase.NOMINAL:
            if sq_rate > 0.25:
                self.current = Phase.DEGRADED
                self._degraded_ticks = 0

        elif self.current == Phase.DEGRADED:
            self._degraded_ticks += 1
            if sq_rate > 0.50:
                self.current = Phase.CRITICAL
            elif sq_rate < 0.12 and ledger_trend >= 0 and self._degraded_ticks > 10:
                self.current = Phase.NOMINAL

        elif self.current == Phase.CRITICAL:
            self.current = Phase.RECOVERY  # automatic

        # RECOVERY: only Closure Auditor can exit
        self.history.append(self.current)
        return self.current

class RecoveryOrchestrator:
    """
    Minimal-cost φ-anchored intervention.
    Blends current state toward 1/3 attractor (accumulator) and φ attractor (velocity).
    Blend rate scales with severity.
    """
    def intervene(self, phase: Phase, ledger: CoherenceLedger,
                  accumulator: float, velocity: float,
                  shortfall: float) -> tuple:

        severity = min(1.0, shortfall / 0.5)  # normalized

        if phase == Phase.DEGRADED:
            blend = 0.05 + 0.10 * severity
            new_acc = (1-blend)*accumulator + blend*GATE_THRESHOLD
            new_vel = velocity
            restore = 0.015

        elif phase in (Phase.CRITICAL, Phase.RECOVERY):
            blend = 0.15 + 0.25 * severity
            new_acc = (1-blend)*accumulator + blend*GATE_THRESHOLD
            new_vel = (1-blend)*velocity    + blend*(PHI % (PI*2))
            restore = 0.04

        else:
            new_acc = accumulator
            new_vel = velocity
            restore = 0.003

        ledger.restore(restore)
        return new_acc % 1.0, new_vel % (PI*2)


class ClosureAuditor:
    """
    Geometric closure enforcement.
    Closure criterion: λ = S_current / S_entry < LAMBDA_REQUIRED
    sustained for SUSTAIN_TICKS consecutive ticks.
    No absolute floor required — contraction itself is the proof of convergence.
    The loop cannot declare itself healthy without passing this test.
    """
    LAMBDA_REQUIRED = 0.95
    SUSTAIN_TICKS   = 12    # must sustain contraction before declaring closure
    MIN_DWELL       = 20    # minimum ticks in Recovery before auditing begins

    def __init__(self):
        self.entry_shortfall: float = 0.0
        self.dwell           = 0
        self.sustain_count   = 0
        self.closure_events: List[int] = []   # ticks where closure confirmed
        self.lambda_history: List[float] = []
        self.escalations     = 0

    def enter_recovery(self, shortfall: float):
        self.entry_shortfall = shortfall
        self.dwell           = 0
        self.sustain_count   = 0

    def audit(self, tick: int, shortfall: float, gate: PhaseGate) -> bool:
        self.dwell += 1

        if self.dwell < self.MIN_DWELL:
            return False

        if self.entry_shortfall < 1e-9:
            gate.current = Phase.NOMINAL
            self.closure_events.append(tick)
            return True

        lam = shortfall / self.entry_shortfall
        self.lambda_history.append(lam)

        if lam < self.LAMBDA_REQUIRED:
            self.sustain_count += 1
        else:
            self.sustain_count = 0  # reset — must be sustained

        if self.sustain_count >= self.SUSTAIN_TICKS:
            # Closure confirmed: contraction sustained
            gate.current = Phase.NOMINAL
            self.closure_events.append(tick)
            return True

        if lam > 1.10 and self.dwell > 30:
            # Recovery diverging — escalate
            gate.current = Phase.CRITICAL
            self.escalations += 1
            self.dwell = 0
            self.sustain_count = 0
            return False

        return False


# ── Main Engine Loop ───────────────────────────────────────────────────

def run_self_evaluating_engine(n_ticks=800):
    ledger   = CoherenceLedger()
    sentinel = ThresholdSentinel(ledger)
    gate     = PhaseGate()
    orch     = RecoveryOrchestrator()
    auditor  = ClosureAuditor()

    accumulator = 0.1
    velocity    = 1.0

    t             = np.arange(n_ticks)
    wave_output   = np.zeros(n_ticks)
    entropy_load  = np.zeros(n_ticks)
    shortfall_sig = np.zeros(n_ticks)
    ledger_log    = np.zeros(n_ticks)
    gate_events   = np.zeros(n_ticks)
    phase_log     = []
    sq_count      = 0
    shortfall_acc = 0.0
    in_recovery   = False

    np.random.seed(42)
    external_data = np.sin(t / 15) + np.random.normal(0, 0.3, n_ticks)

    for i in range(n_ticks):
        inp = external_data[i]

        # Core waveform: π clock × φ depth
        clock_cycle  = np.sin(2 * PI * (i / 10) + accumulator)
        logic_depth  = np.cos(PHI * (accumulator + inp) * velocity)
        current_work = (clock_cycle + logic_depth) / 2
        wave_output[i]  = current_work
        gate_events[i]  = 1.0 if abs(current_work) > GATE_THRESHOLD else 0.0

        # Shortfall measurement
        required      = abs(np.sin(2 * PI * i / 10))
        actual        = abs(current_work)
        raw_sf        = max(0.0, required - actual)
        shortfall_acc = 0.92 * shortfall_acc + 0.08 * raw_sf
        shortfall_sig[i] = shortfall_acc

        # ── Shortfall Engine: self-evaluation ─────────────────────
        sq = sentinel.evaluate(i, shortfall_acc, gate.current)
        if sq:
            sq_count += 1

        prev_phase    = gate.current
        current_phase = gate.evaluate(sentinel.sq_rate(), ledger.trend())
        phase_log.append(current_phase)

        # Detect Recovery entry
        if current_phase == Phase.RECOVERY and not in_recovery:
            in_recovery = True
            auditor.enter_recovery(shortfall_acc)

        # Recovery exit detected
        if in_recovery and current_phase == Phase.NOMINAL:
            in_recovery = False

        # Orchestrator intervention
        if current_phase in (Phase.DEGRADED, Phase.CRITICAL, Phase.RECOVERY):
            accumulator, velocity = orch.intervene(
                current_phase, ledger, accumulator, velocity, shortfall_acc
            )

        # Closure Auditor
        if current_phase == Phase.RECOVERY:
            auditor.audit(i, shortfall_acc, gate)

        # Standard state update
        velocity    = (velocity + (current_work * PHI)) % (PI * 2)
        accumulator = (accumulator + (abs(current_work) / PHI)) % 1.0
        entropy_load[i] = velocity / (PI * 2)
        ledger_log[i]   = ledger.balance

    phase_counts = {p: phase_log.count(p) for p in Phase}
    lam_final = auditor.lambda_history[-1] if auditor.lambda_history else float('nan')
    lam_mean  = float(np.mean(auditor.lambda_history)) if auditor.lambda_history else float('nan')

    return dict(
        t=t, wave=wave_output, shortfall=shortfall_sig,
        entropy=entropy_load, ledger=ledger_log,
        gate_events=gate_events, phase_log=phase_log,
        auditor=auditor,
        metrics=dict(
            efficiency         = 100 - np.mean(entropy_load)*100,
            gate_fire_rate     = np.mean(gate_events)*100,
            mean_shortfall     = np.mean(shortfall_sig),
            total_sq_emitted   = sq_count,
            closure_events     = len(auditor.closure_events),
            escalations        = auditor.escalations,
            lambda_final       = lam_final,
            lambda_mean        = lam_mean,
            converged          = lam_final < auditor.LAMBDA_REQUIRED if not np.isnan(lam_final) else False,
            phase_dist         = {p.value: round(c/n_ticks*100,1) for p,c in phase_counts.items()},
            ledger_final       = ledger_log[-1],
        )
    )


if __name__ == "__main__":
    print("=" * 62)
    print("  ROUTING ENGINE v4 — Self-Evaluating via Shortfall Engine")
    print("=" * 62)
    r  = run_self_evaluating_engine()
    m  = r["metrics"]

    print(f"\n  ROUTING PERFORMANCE")
    print(f"    Steady-State Efficiency   : {m['efficiency']:.2f}%")
    print(f"    1/3 Gate Fire Rate        : {m['gate_fire_rate']:.2f}%")
    print(f"    Mean Shortfall            : {m['mean_shortfall']:.4f}")

    print(f"\n  SHORTFALL ENGINE — SELF-REPORT")
    print(f"    SQs Emitted               : {m['total_sq_emitted']}")
    print(f"    Closure Events (loop closed) : {m['closure_events']}")
    print(f"    Escalations               : {m['escalations']}")
    print(f"    λ final                   : {m['lambda_final']:.4f}  "
          f"({'contracting ✓' if m['converged'] else 'not yet converged'})")
    print(f"    λ mean (over all audits)  : {m['lambda_mean']:.4f}")
    print(f"    Ledger Balance (final)    : {m['ledger_final']:.4f}")

    print(f"\n  PHASE DISTRIBUTION")
    for phase, pct in m['phase_dist'].items():
        filled = int(pct / 2)
        bar    = "█" * filled + "░" * (25 - filled)
        print(f"    {phase:<10}: {pct:5.1f}%  {bar}")

    print(f"\n  CLOSURE AUDIT TRAIL")
    ticks = r['auditor'].closure_events
    if ticks:
        print(f"    Closed at ticks: {ticks}")
    else:
        print(f"    No closures yet — engine still contracting toward convergence")
    print("=" * 62)

  ROUTING ENGINE v4 — Self-Evaluating via Shortfall Engine

  ROUTING PERFORMANCE
    Steady-State Efficiency   : 64.80%
    1/3 Gate Fire Rate        : 43.62%
    Mean Shortfall            : 0.3609

  SHORTFALL ENGINE — SELF-REPORT
    SQs Emitted               : 662
    Closure Events (loop closed) : 72
    Escalations               : 10
    λ final                   : 0.9654  (not yet converged)
    λ mean (over all audits)  : 0.8911
    Ledger Balance (final)    : 1.0000

  PHASE DISTRIBUTION
    NOMINAL   :   4.8%  ██░░░░░░░░░░░░░░░░░░░░░░░
    DEGRADED  :  11.4%  █████░░░░░░░░░░░░░░░░░░░░
    CRITICAL  :   9.1%  ████░░░░░░░░░░░░░░░░░░░░░
    RECOVERY  :  74.8%  █████████████████████████████████████

  CLOSURE AUDIT TRAIL
    Closed at ticks: [91, 94, 97, 100, 231, 234, 237, 240, 261, 264, 267, 270, 273, 276, 324, 327, 330, 333, 336, 339, 342, 345, 348, 351, 354, 357, 360, 399, 402, 405, 408, 411, 414, 417, 420, 423, 426, 450, 453, 456, 459, 462, 465, 468, 471, 474, 477, 480, 505,